# RoFormer: Enhanced Transformer with Rotary Position Embedding (RoPE)

---

**Paper:** [RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864)  
**Authors:** Jianlin Su, Yu Lu, Shengfeng Pan, Ahmed Murtadha, Bo Wen, Yunfeng Liu  
**Affiliation:** Zhuiyi Technology Co., Ltd., Shenzhen, China  
**Published:** 2021 (arXiv: 2104.09864)

---

## Key Contribution

The Transformer architecture (Vaswani et al., 2017) uses self-attention, which is **permutation-equivariant** — it has no inherent notion of token order. This means we must inject position information somehow. RoFormer introduces **Rotary Position Embedding (RoPE)**, a method that encodes **absolute** position through **rotation** of the query and key vectors, such that their dot product naturally depends only on the **relative** distance between tokens.

### Why This Paper Matters

RoPE is now the **dominant** position encoding in modern large language models:
- **LLaMA / LLaMA 2 / LLaMA 3** (Meta)
- **GPT-NeoX / Pythia** (EleutherAI)
- **PaLM / Gemini** (Google)
- **Qwen** (Alibaba)
- **DeepSeek** (DeepSeek AI)
- **Mistral / Mixtral** (Mistral AI)

### What Makes RoPE Special

| Property | Sinusoidal | Learned | Relative (T5/ALiBi) | **RoPE** |
|----------|-----------|---------|---------------------|----------|
| Extra parameters | 0 | $O(L \cdot d)$ | $O(H)$ or 0 | **0** |
| Where position enters | Addition | Addition | Attention bias | **Multiplication** |
| Captures relative pos. | ✗ | ✗ | ✓ | **✓** |
| Extrapolates to longer seqs | Partially | ✗ | ✓ | **✓** |
| Theoretically derived | ✗ (guessed) | ✗ | ✗ | **✓** |

In this notebook we will:
1. Derive RoPE from first principles (the relativity constraint)
2. Build up from the 2D case to the general $d$-dimensional case
3. Implement the efficient version used in production LLMs
4. Visualize its frequency structure and long-range decay
5. Show it working inside a full attention layer
6. Demonstrate length extrapolation

## 2. Setup & Imports

### WHAT
We import the core libraries needed throughout this notebook.

### WHY
- **PyTorch** — for tensor operations and building neural network modules
- **matplotlib** — for plotting frequency ladders, decay curves, attention maps
- **numpy** — for numerical utilities
- **math** — for `sqrt`, `pi`, etc.

### HOW
Standard imports. We set seeds for reproducibility and configure matplotlib for inline display.

### WHERE
Prerequisite for all subsequent cells.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math
from typing import Tuple, Optional

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Matplotlib styling for clean figures
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Check device (RoPE runs fine on CPU for our demos)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 3. The Position Encoding Problem

### WHAT
We demonstrate **why** self-attention is position-agnostic and **why** that's a problem.

### WHY
The self-attention mechanism computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

If we **permute** the input sequence $\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_n$, the output is the **same permutation** of the original output. The mechanism treats the input as a **set**, not a **sequence**. But language has order — "dog bites man" ≠ "man bites dog".

### HOW
We create a simple attention computation and show that permuting the input permutes the output identically — proving position-agnosticism.

### WHERE
Section 1 of the paper: *"Since the self-attention in Transformer is permutation invariant, positional encoding is essential for a Transformer-based language model."*

### History of Position Encoding Approaches

1. **Sinusoidal (Vaswani et al., 2017):** $PE_{(pos,2i)} = \sin(pos / 10000^{2i/d})$ — fixed, added to embeddings. No extra parameters but doesn't truly capture relative positions.

2. **Learned absolute (BERT, GPT-2):** A lookup table $E_{pos} \in \mathbb{R}^{L_{max} \times d}$. Cannot extrapolate beyond $L_{max}$.

3. **Relative (Shaw et al., 2018; T5, ALiBi):** Modify attention scores by relative distance. Adds complexity to the attention kernel.

4. **RoPE (this paper):** Encode position by **rotating** $q$ and $k$ vectors. The dot product automatically depends on $m - n$.

In [ ]:
# === Demonstrate: Self-Attention is Permutation-Equivariant ===

def simple_attention(x: torch.Tensor) -> torch.Tensor:
    """Compute self-attention WITHOUT any position encoding.
    x: (seq_len, d_model)
    """
    d_k = x.shape[-1]
    # Q = K = V = x for simplicity (no learned projections)
    scores = torch.matmul(x, x.transpose(-2, -1)) / math.sqrt(d_k)  # (seq_len, seq_len)
    attn_weights = F.softmax(scores, dim=-1)                          # (seq_len, seq_len)
    output = torch.matmul(attn_weights, x)                            # (seq_len, d_model)
    return output

# Create a small sequence: 4 tokens, each with dimension 8
seq_len, d_model = 4, 8
x = torch.randn(seq_len, d_model)

# Original output
out_original = simple_attention(x)

# Permute: swap positions 0↔2 and 1↔3
perm = [2, 3, 0, 1]
x_permuted = x[perm]  # Apply the permutation to rows

# Output on permuted input
out_permuted = simple_attention(x_permuted)

# The permuted output should equal the original output with the SAME permutation applied
out_original_repermuted = out_original[perm]

diff = (out_permuted - out_original_repermuted).abs().max().item()
print(f"Max difference between permuted outputs: {diff:.2e}")
print(f"Permutation equivariant? {diff < 1e-6}")
print()
print("This proves self-attention treats the input as a SET, not a SEQUENCE.")
print("Without position encoding, 'dog bites man' and 'man bites dog' produce")
print("the same attention pattern (just permuted).")

## 4. The Relativity Constraint — Eq. 11

### WHAT
We formalize the **key requirement** that RoPE must satisfy: the dot product between a query at position $m$ and a key at position $n$ should depend only on the **token content** and the **relative offset** $m - n$, not on the absolute positions $m$ and $n$ individually.

### WHY
If position information enters via functions $f_q$ and $f_k$ that transform the raw embeddings:

$$\mathbf{q}_m = f_q(\mathbf{x}_m, m), \quad \mathbf{k}_n = f_k(\mathbf{x}_n, n)$$

Then we want the attention logit (their inner product) to satisfy:

$$\boxed{\langle f_q(\mathbf{x}_m, m),\; f_k(\mathbf{x}_n, n) \rangle = g(\mathbf{x}_m, \mathbf{x}_n, m - n)} \quad \text{(Eq. 11)}$$

for some function $g$. This is the **relativity constraint**: the inner product is a function of content ($\mathbf{x}_m$, $\mathbf{x}_n$) and **relative distance** ($m - n$), but NOT of absolute position.

### HOW
We demonstrate that **additive** position encodings (like learned or sinusoidal) **violate** this constraint. When you shift both $m$ and $n$ by the same amount $\Delta$, the dot product changes for additive encodings but should remain the same under the relativity constraint.

### WHERE
Section 3.2 of the paper, Equation 11. The authors write: *"We want to find f_q and f_k such that the inner product between query and key only depends on the relative position m − n."*

In [ ]:
# === Demonstrate: Additive position encodings VIOLATE the relativity constraint ===

def sinusoidal_pe(pos: int, d_model: int) -> torch.Tensor:
    """Generate sinusoidal position encoding for a single position.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
    """
    pe = torch.zeros(d_model)
    for i in range(0, d_model, 2):
        freq = 1.0 / (10000.0 ** (i / d_model))   # Frequency for this dimension pair
        pe[i]     = math.sin(pos * freq)            # Even dimensions: sin
        pe[i + 1] = math.cos(pos * freq)            # Odd dimensions: cos
    return pe

d = 16  # Embedding dimension

# Token embeddings (content)
x_m = torch.randn(d)  # Token at position m
x_n = torch.randn(d)  # Token at position n

# Weight matrices for Q and K projections
W_q = torch.randn(d, d) * 0.1
W_k = torch.randn(d, d) * 0.1

# === Additive position encoding ===
# f_q(x, m) = W_q * (x + PE(m)), f_k(x, n) = W_k * (x + PE(n))

def compute_additive_dot(x_m, x_n, m, n, W_q, W_k):
    """Dot product with additive position encoding."""
    q = W_q @ (x_m + sinusoidal_pe(m, len(x_m)))  # Add PE to embedding, then project
    k = W_k @ (x_n + sinusoidal_pe(n, len(x_n)))
    return torch.dot(q, k)

# Test the relativity constraint:
# If <f_q(x_m, m), f_k(x_n, n)> = g(x_m, x_n, m-n),
# then shifting both positions by Δ should NOT change the result.

m, n = 5, 2   # Relative distance = 3
delta = 10     # Shift both by 10

dot_original = compute_additive_dot(x_m, x_n, m, n, W_q, W_k)
dot_shifted  = compute_additive_dot(x_m, x_n, m + delta, n + delta, W_q, W_k)

print("=== Testing Relativity Constraint with Additive PE ===")
print(f"Positions (m={m}, n={n}), relative distance = {m-n}")
print(f"Dot product: {dot_original.item():.6f}")
print()
print(f"Shifted positions (m={m+delta}, n={n+delta}), relative distance = {m+delta-n-delta}")
print(f"Dot product: {dot_shifted.item():.6f}")
print()
print(f"Difference: {abs(dot_original.item() - dot_shifted.item()):.6f}")
print(f"Relativity constraint satisfied? {abs(dot_original.item() - dot_shifted.item()) < 1e-5}")
print()
print("CONCLUSION: Additive position encodings VIOLATE the relativity constraint.")
print("The dot product depends on absolute positions, not just relative distance.")

## 5. The 2D Solution — Eq. 12

### WHAT
We derive the solution to the relativity constraint in the simplest case: **2-dimensional** embeddings. The solution uses **complex number multiplication** (equivalently, 2D rotation).

### WHY
The paper starts with $d = 2$ to build intuition. In 2D, every vector $\mathbf{x} = (x_1, x_2)$ can be written as a complex number $z = x_1 + ix_2$. The key insight is that **multiplying by $e^{i\theta}$ rotates** the vector by angle $\theta$, and this rotation has the magical property that:

$$\text{Re}[(z_1 e^{im\theta}) \cdot \overline{(z_2 e^{in\theta})}] = \text{Re}[z_1 \bar{z}_2 \cdot e^{i(m-n)\theta}]$$

The product **only depends on** $m - n$, not on $m$ and $n$ separately!

### HOW
The paper defines (Eq. 12):

$$f_q(\mathbf{x}_m, m) = (W_q \mathbf{x}_m) e^{im\theta}$$
$$f_k(\mathbf{x}_n, n) = (W_k \mathbf{x}_n) e^{in\theta}$$

where $W_q \mathbf{x}_m$ and $W_k \mathbf{x}_n$ are treated as complex numbers.

Then the inner product becomes:

$$\langle f_q, f_k \rangle = \text{Re}\Big[(W_q \mathbf{x}_m) \overline{(W_k \mathbf{x}_n)} \cdot e^{i(m-n)\theta}\Big]$$

This is $g(\mathbf{x}_m, \mathbf{x}_n, m-n)$ — the relativity constraint is satisfied! ✓

### WHERE
Section 3.3 of the paper, Equations 12-13. The authors formulate: *"With the help of 2D case, we first find the solution of $f$ using the complex number representation."*

In [ ]:
# === 2D Rotation via Complex Multiplication ===

def apply_rope_2d_complex(x: torch.Tensor, position: int, theta: float) -> torch.Tensor:
    """Apply RoPE to a 2D vector using complex number multiplication.
    
    x:        (2,) tensor representing a 2D vector [x1, x2]
    position: the absolute position m
    theta:    the base frequency θ
    
    Returns: rotated 2D vector = x * e^{i*m*θ}
    """
    # Convert to complex: z = x[0] + i*x[1]
    z = torch.complex(x[0], x[1])
    
    # The rotation factor: e^{i*m*θ} = cos(mθ) + i*sin(mθ)
    angle = position * theta
    rotation = torch.complex(
        torch.tensor(math.cos(angle)),
        torch.tensor(math.sin(angle))
    )
    
    # Multiply: this rotates z by angle (m*θ)
    z_rotated = z * rotation
    
    # Convert back to real 2D vector
    return torch.tensor([z_rotated.real, z_rotated.imag])


def dot_product_2d(v1: torch.Tensor, v2: torch.Tensor) -> float:
    """Standard dot product of two 2D vectors."""
    return torch.dot(v1, v2).item()


# === Verify the relativity constraint for RoPE in 2D ===
theta = 1.0  # Base frequency

# Raw query and key vectors (after W_q, W_k projection)
q_raw = torch.tensor([1.5, 0.8])   # Some query vector
k_raw = torch.tensor([0.3, -1.2])  # Some key vector

# Test 1: positions m=5, n=2 → relative distance = 3
q_rotated_1 = apply_rope_2d_complex(q_raw, position=5, theta=theta)
k_rotated_1 = apply_rope_2d_complex(k_raw, position=2, theta=theta)
dot_1 = dot_product_2d(q_rotated_1, k_rotated_1)

# Test 2: positions m=15, n=12 → relative distance = 3 (same!)
q_rotated_2 = apply_rope_2d_complex(q_raw, position=15, theta=theta)
k_rotated_2 = apply_rope_2d_complex(k_raw, position=12, theta=theta)
dot_2 = dot_product_2d(q_rotated_2, k_rotated_2)

# Test 3: positions m=100, n=97 → relative distance = 3 (same!)
q_rotated_3 = apply_rope_2d_complex(q_raw, position=100, theta=theta)
k_rotated_3 = apply_rope_2d_complex(k_raw, position=97, theta=theta)
dot_3 = dot_product_2d(q_rotated_3, k_rotated_3)

# Test 4: different relative distance m=5, n=1 → relative distance = 4
q_rotated_4 = apply_rope_2d_complex(q_raw, position=5, theta=theta)
k_rotated_4 = apply_rope_2d_complex(k_raw, position=1, theta=theta)
dot_4 = dot_product_2d(q_rotated_4, k_rotated_4)

print("=== Verifying Relativity Constraint for RoPE (2D) ===")
print(f"")
print(f"m=5,   n=2   → relative distance = 3 → dot = {dot_1:.8f}")
print(f"m=15,  n=12  → relative distance = 3 → dot = {dot_2:.8f}")
print(f"m=100, n=97  → relative distance = 3 → dot = {dot_3:.8f}")
print(f"")
print(f"All equal? {abs(dot_1 - dot_2) < 1e-5 and abs(dot_2 - dot_3) < 1e-5}  ✓")
print(f"")
print(f"m=5,   n=1   → relative distance = 4 → dot = {dot_4:.8f}")
print(f"Different from distance=3? {abs(dot_1 - dot_4) > 1e-5}  ✓")
print()
print("CONCLUSION: RoPE's dot product depends ONLY on (m-n), not on m or n individually!")

In [ ]:
# === Visualization: Rotation in the 2D Complex Plane ===

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Show how a single vector rotates through positions ---
ax = axes[0]
theta = 0.5  # Base frequency
v = torch.tensor([1.0, 0.3])  # Original vector

positions = range(0, 13)
colors = plt.cm.viridis(np.linspace(0, 1, len(positions)))

for pos, color in zip(positions, colors):
    v_rot = apply_rope_2d_complex(v, position=pos, theta=theta)
    ax.annotate('', xy=(v_rot[0].item(), v_rot[1].item()), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    ax.plot(v_rot[0].item(), v_rot[1].item(), 'o', color=color, markersize=5)
    if pos % 3 == 0:  # Label every 3rd position
        ax.annotate(f'm={pos}', (v_rot[0].item() * 1.1, v_rot[1].item() * 1.1),
                   fontsize=8, color=color)

# Draw the unit circle for reference
circle_t = np.linspace(0, 2 * np.pi, 100)
radius = torch.norm(v).item()  # The rotation preserves the vector's magnitude
ax.plot(radius * np.cos(circle_t), radius * np.sin(circle_t), 'k--', alpha=0.2)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.set_xlabel('Dimension 1 (Real)')
ax.set_ylabel('Dimension 2 (Imaginary)')
ax.set_title(f'RoPE rotates vectors through positions\n(θ={theta})')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)

# --- Right plot: Dot product vs relative distance ---
ax = axes[1]
theta = 0.3
q_raw = torch.tensor([1.0, 0.5])
k_raw = torch.tensor([0.8, -0.3])

# Fix m=0, vary n to change relative distance
distances = list(range(-20, 21))
dots_from_pos0 = []   # Starting from m=0
dots_from_pos50 = []  # Starting from m=50 (same relative distances)

for d_rel in distances:
    # From position 0
    q_rot = apply_rope_2d_complex(q_raw, position=0, theta=theta)
    k_rot = apply_rope_2d_complex(k_raw, position=-d_rel, theta=theta)  # n = m - d_rel
    dots_from_pos0.append(dot_product_2d(q_rot, k_rot))
    
    # From position 50
    q_rot = apply_rope_2d_complex(q_raw, position=50, theta=theta)
    k_rot = apply_rope_2d_complex(k_raw, position=50 - d_rel, theta=theta)
    dots_from_pos50.append(dot_product_2d(q_rot, k_rot))

ax.plot(distances, dots_from_pos0, 'b-', linewidth=2, label='m=0')
ax.plot(distances, dots_from_pos50, 'r--', linewidth=2, label='m=50')
ax.set_xlabel('Relative distance (m - n)')
ax.set_ylabel('Dot product ⟨q_m, k_n⟩')
ax.set_title('Dot product depends ONLY on relative distance')
ax.legend()

plt.tight_layout()
plt.show()
print("Left: Each arrow is the same vector rotated to a different position.")
print("Right: The blue and red curves overlap perfectly — dot product depends only on (m-n).")

## 6. The Rotation Matrix — Eq. 14

### WHAT
We express the complex multiplication from Section 5 as an explicit **2D rotation matrix**. This makes the connection to geometry crystal clear.

### WHY
Complex multiplication by $e^{im\theta}$ is equivalent to multiplying by the rotation matrix:

$$R(m\theta) = \begin{pmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{pmatrix}$$

This is the standard 2D rotation matrix from linear algebra. The paper uses this to generalize RoPE to higher dimensions.

### HOW
The position-encoded query at position $m$ is (Eq. 14):

$$f_q(\mathbf{x}_m, m) = R(m\theta) \cdot W_q \mathbf{x}_m$$

The dot product between query $m$ and key $n$:

$$\langle f_q, f_k \rangle = (W_q \mathbf{x}_m)^\top R(m\theta)^\top R(n\theta) (W_k \mathbf{x}_n)$$

Since rotation matrices satisfy $R(\alpha)^\top R(\beta) = R(\beta - \alpha)$:

$$= (W_q \mathbf{x}_m)^\top R((n - m)\theta) (W_k \mathbf{x}_n)$$

This depends only on $n - m$ (relative position)! ✓

### WHERE
Section 3.3, Equation 14. The paper states: *"[This is] equivalent to a rotation of the 2D vector."*

In [ ]:
# === 2D Rotation via Explicit Matrix ===

def rotation_matrix_2d(angle: float) -> torch.Tensor:
    """Build the 2D rotation matrix for a given angle.
    
    R(θ) = [[cos θ, -sin θ],
            [sin θ,  cos θ]]
    """
    c = math.cos(angle)
    s = math.sin(angle)
    return torch.tensor([[c, -s],
                         [s,  c]])


def apply_rope_2d_matrix(x: torch.Tensor, position: int, theta: float) -> torch.Tensor:
    """Apply RoPE to a 2D vector using explicit rotation matrix."""
    R = rotation_matrix_2d(position * theta)  # R(m*θ)
    return R @ x                               # Matrix-vector multiply


# === Verify: Complex multiplication == Matrix multiplication ===
theta = 0.7
v = torch.tensor([1.3, -0.6])
pos = 5

# Method 1: Complex multiplication (from Section 5)
v_complex = apply_rope_2d_complex(v, position=pos, theta=theta)

# Method 2: Matrix multiplication (this section)
v_matrix = apply_rope_2d_matrix(v, position=pos, theta=theta)

print("=== Verifying Equivalence: Complex vs Matrix ===")
print(f"Input vector: {v.tolist()}")
print(f"Position: {pos}, θ = {theta}")
print(f"")
print(f"Complex method: [{v_complex[0]:.8f}, {v_complex[1]:.8f}]")
print(f"Matrix method:  [{v_matrix[0]:.8f}, {v_matrix[1]:.8f}]")
print(f"")
diff = (v_complex - v_matrix).abs().max().item()
print(f"Max difference: {diff:.2e}")
print(f"Equivalent? {diff < 1e-6}  ✓")

# === Verify the rotation matrix property: R(α)^T @ R(β) = R(β - α) ===
alpha, beta = 2.0, 5.0
R_alpha = rotation_matrix_2d(alpha)
R_beta = rotation_matrix_2d(beta)
R_diff = rotation_matrix_2d(beta - alpha)

product = R_alpha.T @ R_beta    # Should equal R(β - α)
diff_matrices = (product - R_diff).abs().max().item()
print(f"\nR(α)^T @ R(β) = R(β - α)? Error = {diff_matrices:.2e}  ✓")
print("This is WHY the dot product depends only on (m-n)!")

## 7. General $d$-Dimensional Form — Eq. 15-16

### WHAT
We generalize RoPE from 2D to arbitrary (even) dimension $d$. Instead of a single rotation, we perform $d/2$ **independent** 2D rotations, each in its own plane and with its own frequency.

### WHY
In practice, Transformer head dimensions are $d = 64, 128$, etc. We need RoPE to work in these higher dimensions. The key insight is that a $d$-dimensional vector can be split into $d/2$ pairs, and each pair gets its own 2D rotation.

### HOW
The general rotation matrix is **block-diagonal** (Eq. 15):

$$R^d_{\Theta, m} = \begin{pmatrix}
\cos m\theta_1 & -\sin m\theta_1 & 0 & 0 & \cdots & 0 & 0 \\
\sin m\theta_1 & \cos m\theta_1 & 0 & 0 & \cdots & 0 & 0 \\
0 & 0 & \cos m\theta_2 & -\sin m\theta_2 & \cdots & 0 & 0 \\
0 & 0 & \sin m\theta_2 & \cos m\theta_2 & \cdots & 0 & 0 \\
\vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots \\
0 & 0 & 0 & 0 & \cdots & \cos m\theta_{d/2} & -\sin m\theta_{d/2} \\
0 & 0 & 0 & 0 & \cdots & \sin m\theta_{d/2} & \cos m\theta_{d/2}
\end{pmatrix}$$

The **frequency schedule** is (Eq. 16):

$$\theta_i = 10000^{-2(i-1)/d}, \quad i = 1, 2, \ldots, d/2$$

This is the same geometric frequency schedule as the original sinusoidal encoding, but used multiplicatively instead of additively.

**Frequency intuition:**
- $\theta_1 = 1$ (fastest rotation, encodes nearby positions)
- $\theta_{d/2} = 10000^{-1} = 0.0001$ (slowest rotation, encodes long-range positions)

### WHERE
Section 3.3, Equations 15-16. The paper states: *"We divide the d-dim space into d/2 subspaces and combine them in the form of the rotation matrix."*

In [ ]:
# === Build the General d-dimensional RoPE Rotation Matrix ===

def compute_theta_schedule(d: int, base: float = 10000.0) -> torch.Tensor:
    """Compute the frequency schedule θ_i = base^{-2(i-1)/d}.
    
    d:    head dimension (must be even)
    base: frequency base (default 10000 from the paper)
    
    Returns: (d/2,) tensor of frequencies
    """
    assert d % 2 == 0, "d must be even"
    # i goes from 0 to d/2 - 1, so exponent = -2*i/d = 2i/d negated
    i = torch.arange(0, d // 2, dtype=torch.float32)  # [0, 1, 2, ..., d/2-1]
    theta = base ** (-2.0 * i / d)                       # Geometric decay
    return theta


def build_rope_rotation_matrix(position: int, d: int, base: float = 10000.0) -> torch.Tensor:
    """Build the full d×d block-diagonal rotation matrix R^d_{Θ,m}.
    
    position: the sequence position m
    d:        the dimension (must be even)
    base:     frequency base
    
    Returns: (d, d) rotation matrix
    """
    thetas = compute_theta_schedule(d, base)  # (d/2,)
    R = torch.zeros(d, d)                     # Initialize d×d zero matrix
    
    for i in range(d // 2):
        angle = position * thetas[i]  # m * θ_i
        c = torch.cos(angle)
        s = torch.sin(angle)
        # Fill the 2×2 block at position (2i, 2i)
        R[2*i,   2*i]   = c    # cos
        R[2*i,   2*i+1] = -s   # -sin
        R[2*i+1, 2*i]   = s    # sin
        R[2*i+1, 2*i+1] = c    # cos
    
    return R


# === Build and visualize for d=8 ===
d = 8
position = 5

R = build_rope_rotation_matrix(position, d)

print(f"=== RoPE Rotation Matrix R^{d}_{{Θ,{position}}} ===")
print(f"Shape: {R.shape}")
print(f"\nFrequency schedule θ_i for d={d}:")
thetas = compute_theta_schedule(d)
for i, t in enumerate(thetas):
    print(f"  θ_{i+1} = {t.item():.6f}  (period = {2*math.pi/t.item():.1f} positions)")

print(f"\nRotation matrix (rounded to 3 decimals):")
print(R.numpy().round(3))

# Verify it's orthogonal: R^T @ R = I
identity_check = R.T @ R
identity_error = (identity_check - torch.eye(d)).abs().max().item()
print(f"\nOrthogonality check: ||R^T R - I||_max = {identity_error:.2e}  (should be ~0)")

# Verify it preserves norms
v = torch.randn(d)
v_rotated = R @ v
print(f"Norm preservation: ||v|| = {v.norm():.6f}, ||Rv|| = {v_rotated.norm():.6f}")

In [ ]:
# === Heatmap Visualization of the Block-Diagonal Structure ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Different dimensions to show the block structure
for ax, dim in zip(axes, [4, 8, 16]):
    R = build_rope_rotation_matrix(position=3, d=dim)
    im = ax.imshow(R.numpy(), cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
    ax.set_title(f'd = {dim}\n({dim//2} rotation planes)', fontsize=13)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    
    # Grid lines to highlight the 2×2 blocks
    for i in range(0, dim, 2):
        ax.axhline(y=i - 0.5, color='gray', linewidth=0.5)
        ax.axvline(x=i - 0.5, color='gray', linewidth=0.5)

plt.colorbar(im, ax=axes, label='Matrix value', shrink=0.8)
plt.suptitle('RoPE Rotation Matrix: Block-Diagonal Structure (Eq. 15)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Each 2×2 block on the diagonal is an independent rotation.")
print("Off-diagonal blocks are all zeros → the rotation planes are orthogonal.")

## 8. The Efficient Implementation — Eq. 34

### WHAT
We implement the **production-quality** version of RoPE that avoids constructing the full $d \times d$ rotation matrix. Instead, it uses element-wise operations.

### WHY
The full matrix multiply is $O(d^2)$ per position. The efficient version is $O(d)$ — a massive speedup. This is the version used in **every modern LLM** (LLaMA, GPT-NeoX, etc.).

### HOW
The key identity (Eq. 34 in the paper) is:

$$R^d_{\Theta,m} \mathbf{x} = \underbrace{\begin{pmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \\ \vdots \\ x_{d-1} \\ x_d \end{pmatrix}}_{\mathbf{x}} \otimes \underbrace{\begin{pmatrix} \cos m\theta_1 \\ \cos m\theta_1 \\ \cos m\theta_2 \\ \cos m\theta_2 \\ \vdots \\ \cos m\theta_{d/2} \\ \cos m\theta_{d/2} \end{pmatrix}}_{\cos \text{ part}} + \underbrace{\begin{pmatrix} -x_2 \\ x_1 \\ -x_4 \\ x_3 \\ \vdots \\ -x_d \\ x_{d-1} \end{pmatrix}}_{\text{rotate\_half}(\mathbf{x})} \otimes \underbrace{\begin{pmatrix} \sin m\theta_1 \\ \sin m\theta_1 \\ \sin m\theta_2 \\ \sin m\theta_2 \\ \vdots \\ \sin m\theta_{d/2} \\ \sin m\theta_{d/2} \end{pmatrix}}_{\sin \text{ part}}$$

where $\otimes$ is **element-wise** multiplication (Hadamard product).

The `rotate_half` operation swaps adjacent pairs and negates every other element: $(x_1, x_2, x_3, x_4) \to (-x_2, x_1, -x_4, x_3)$.

This can be verified by expanding the 2×2 block multiplication:
$$\begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} = \begin{pmatrix} x_1 \cos\theta - x_2 \sin\theta \\ x_1 \sin\theta + x_2 \cos\theta \end{pmatrix} = \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} \odot \begin{pmatrix} \cos\theta \\ \cos\theta \end{pmatrix} + \begin{pmatrix} -x_2 \\ x_1 \end{pmatrix} \odot \begin{pmatrix} \sin\theta \\ \sin\theta \end{pmatrix}$$

### WHERE
Section 3.4, Equation 34. The paper calls this the *"computationally efficient form"*.

In [ ]:
# === The Efficient RoPE Implementation (what LLMs actually use) ===

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Rotate adjacent pairs: (x1, x2, x3, x4, ...) → (-x2, x1, -x4, x3, ...)
    
    x: (..., d) where d is even
    """
    # Split into first-half and second-half of each pair
    x1 = x[..., ::2]   # Even indices: x1, x3, x5, ...  shape (..., d/2)
    x2 = x[..., 1::2]  # Odd indices:  x2, x4, x6, ...  shape (..., d/2)
    
    # Interleave: (-x2, x1, -x4, x3, ...)
    # We need to reconstruct the full tensor with the swapped pairs
    rotated = torch.stack((-x2, x1), dim=-1)  # (..., d/2, 2)
    return rotated.reshape(x.shape)             # (..., d)


def precompute_freqs(d: int, max_seq_len: int, base: float = 10000.0) -> Tuple[torch.Tensor, torch.Tensor]:
    """Precompute cos and sin tables for all positions up to max_seq_len.
    
    d:           head dimension (must be even)
    max_seq_len: maximum sequence length
    base:        frequency base
    
    Returns:
        cos_table: (max_seq_len, d) - cosine values, each θ_i repeated twice
        sin_table: (max_seq_len, d) - sine values, each θ_i repeated twice
    """
    # Step 1: Compute the d/2 frequencies
    theta = compute_theta_schedule(d, base)  # (d/2,)
    
    # Step 2: Create position indices
    positions = torch.arange(max_seq_len, dtype=torch.float32)  # (max_seq_len,)
    
    # Step 3: Outer product → angles for every (position, frequency) pair
    # angles[m, i] = m * θ_i
    angles = torch.outer(positions, theta)  # (max_seq_len, d/2)
    
    # Step 4: Each frequency is used for a PAIR of dimensions,
    # so we repeat each angle: [mθ_1, mθ_1, mθ_2, mθ_2, ...]
    angles = angles.repeat(1, 2)  # WRONG - this concatenates, not interleaves
    # Actually we need interleaving:
    angles = torch.stack([angles[:, :d//2], angles[:, :d//2]], dim=-1).reshape(max_seq_len, d)
    
    cos_table = torch.cos(angles)  # (max_seq_len, d)
    sin_table = torch.sin(angles)  # (max_seq_len, d)
    
    return cos_table, sin_table


def apply_rotary_emb_efficient(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply RoPE using the efficient element-wise formulation (Eq. 34).
    
    x:   (..., seq_len, d)
    cos: (seq_len, d) or broadcastable
    sin: (seq_len, d) or broadcastable
    
    Returns: (..., seq_len, d) rotated tensor
    """
    # Eq. 34: x * cos(mθ) + rotate_half(x) * sin(mθ)
    return x * cos + rotate_half(x) * sin


# === Verify: Efficient Implementation == Full Matrix Multiply ===
d = 8
x = torch.randn(d)  # A single vector
position = 7

# Method 1: Full matrix multiply (Section 7)
R = build_rope_rotation_matrix(position, d)
result_matrix = R @ x

# Method 2: Efficient element-wise (this section)
cos_table, sin_table = precompute_freqs(d, max_seq_len=position + 1)
cos_m = cos_table[position]  # (d,) cos values at position m
sin_m = sin_table[position]  # (d,) sin values at position m
result_efficient = apply_rotary_emb_efficient(x, cos_m, sin_m)

print("=== Verifying Efficient Implementation (Eq. 34) ===")
print(f"Input x:          {x.numpy().round(4)}")
print(f"Matrix multiply:  {result_matrix.numpy().round(4)}")
print(f"Efficient (Eq34): {result_efficient.numpy().round(4)}")
print(f"")
diff = (result_matrix - result_efficient).abs().max().item()
print(f"Max difference: {diff:.2e}")
print(f"Equivalent? {diff < 1e-5}  ✓")
print(f"")
print(f"Computational cost comparison:")
print(f"  Matrix multiply: O(d²) = O({d**2})")
print(f"  Efficient:       O(d)  = O({d})")
print(f"  Speedup: {d}x for d={d}, would be {128}x for d=128")

In [ ]:
# === Batched version that handles full sequences ===
# This is closer to what actually runs in a Transformer

def precompute_freqs_cis(d: int, max_seq_len: int, base: float = 10000.0) -> torch.Tensor:
    """Precompute RoPE frequencies as complex exponentials (LLaMA-style).
    
    This uses the complex number representation directly:
    freqs_cis[m, i] = e^{i*m*θ_i} = cos(m*θ_i) + i*sin(m*θ_i)
    
    d:           head dimension (must be even)
    max_seq_len: max positions to precompute
    base:        frequency base
    
    Returns: (max_seq_len, d/2) complex tensor
    """
    # Frequencies: θ_i = base^{-2i/d} for i = 0, 1, ..., d/2-1
    theta = 1.0 / (base ** (torch.arange(0, d, 2, dtype=torch.float32) / d))
    
    # Position indices: [0, 1, 2, ..., max_seq_len-1]
    t = torch.arange(max_seq_len, dtype=torch.float32)
    
    # Outer product: angles[m, i] = m * θ_i
    angles = torch.outer(t, theta)  # (max_seq_len, d/2)
    
    # Convert to complex: e^{i*angle} = cos(angle) + i*sin(angle)
    freqs_cis = torch.polar(torch.ones_like(angles), angles)  # (max_seq_len, d/2)
    
    return freqs_cis


def apply_rotary_emb_complex(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """Apply RoPE using complex multiplication (LLaMA-style).
    
    x:         (batch, seq_len, n_heads, d) real tensor
    freqs_cis: (seq_len, d/2) complex tensor
    
    Returns:   (batch, seq_len, n_heads, d) rotated tensor
    """
    # Reshape x to complex: pair up adjacent dimensions
    # (batch, seq_len, n_heads, d) → (batch, seq_len, n_heads, d/2, 2)
    x_reshaped = x.float().reshape(*x.shape[:-1], -1, 2)
    
    # Convert to complex: each pair (a, b) becomes a + ib
    x_complex = torch.view_as_complex(x_reshaped)  # (..., d/2)
    
    # Reshape freqs_cis for broadcasting: (1, seq_len, 1, d/2)
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2)  # Add batch and head dims
    
    # Complex multiplication = rotation!
    x_rotated = x_complex * freqs_cis  # (..., d/2) complex
    
    # Convert back to real: (a + ib) → (a, b)
    x_out = torch.view_as_real(x_rotated)  # (..., d/2, 2)
    
    return x_out.reshape(x.shape)  # (..., d)


# === Test the batched complex implementation ===
batch, seq_len, n_heads, d = 2, 10, 4, 8
x = torch.randn(batch, seq_len, n_heads, d)
freqs = precompute_freqs_cis(d, seq_len)

x_rotated = apply_rotary_emb_complex(x, freqs)

# Verify for a single element against the matrix method
b_idx, s_idx, h_idx = 0, 5, 2
single_vec = x[b_idx, s_idx, h_idx]  # (d,)
R = build_rope_rotation_matrix(s_idx, d)
expected = R @ single_vec
actual = x_rotated[b_idx, s_idx, h_idx]

diff = (expected - actual).abs().max().item()
print(f"=== Batched Complex Implementation ===")
print(f"Input shape:  {x.shape}  (batch, seq_len, n_heads, d)")
print(f"Output shape: {x_rotated.shape}")
print(f"")
print(f"Spot check at [batch=0, pos=5, head=2]:")
print(f"  Matrix method:  {expected.numpy().round(4)}")
print(f"  Complex method: {actual.detach().numpy().round(4)}")
print(f"  Max difference: {diff:.2e}  ✓")

## 9. Frequency Ladder Visualization

### WHAT
We visualize the $d/2$ different frequency components of RoPE to understand **how different dimensions encode different ranges of positional information**.

### WHY
RoPE uses a geometric sequence of frequencies: $\theta_i = 10000^{-2(i-1)/d}$. This means:
- **Low-index dimensions** (small $i$) rotate **fast** → encode **nearby** positions
- **High-index dimensions** (large $i$) rotate **slowly** → encode **far-apart** positions

This is analogous to how digits in a number work: the ones digit changes fast (nearby resolution) while the thousands digit changes slowly (long-range resolution).

### HOW
We plot $\cos(m \cdot \theta_i)$ as a function of position $m$ for each rotation plane $i$.

### WHERE
Section 3.3 discusses the frequency schedule. The geometric decay is borrowed from the original sinusoidal PE (Vaswani et al., 2017), but is applied multiplicatively in RoPE.

In [ ]:
# === Frequency Ladder: How Different Dimensions Encode Position ===

d = 64  # Typical head dimension
max_pos = 512
thetas = compute_theta_schedule(d)  # (d/2,) = (32,)

positions = torch.arange(max_pos, dtype=torch.float32)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- Top plot: cos(m * θ_i) for selected frequency planes ---
ax = axes[0]
planes_to_show = [0, 4, 8, 16, 24, 31]  # Select diverse planes
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(planes_to_show)))

for plane_idx, color in zip(planes_to_show, colors):
    theta_i = thetas[plane_idx].item()
    cos_values = torch.cos(positions * theta_i)
    period = 2 * math.pi / theta_i
    ax.plot(positions.numpy(), cos_values.numpy(), color=color, linewidth=1.5,
            label=f'Plane {plane_idx}: θ={theta_i:.4f}, period={period:.0f}')

ax.set_xlabel('Position m')
ax.set_ylabel('cos(m · θᵢ)')
ax.set_title(f'Frequency Ladder: cos(m·θᵢ) for d={d} ({d//2} rotation planes)', fontsize=14)
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, max_pos)

# --- Bottom plot: 2D heatmap of all frequencies ---
ax = axes[1]
angles = torch.outer(positions, thetas)  # (max_pos, d/2)
cos_map = torch.cos(angles).numpy()       # (max_pos, d/2)

im = ax.imshow(cos_map.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1,
               extent=[0, max_pos, d//2 - 0.5, -0.5])
ax.set_xlabel('Position m')
ax.set_ylabel('Rotation plane index i')
ax.set_title('Full Frequency Heatmap: cos(m·θᵢ) across all planes', fontsize=14)
plt.colorbar(im, ax=ax, label='cos(m·θᵢ)')

plt.tight_layout()
plt.show()

print("Top: Low-index planes oscillate FAST (fine position resolution).")
print("     High-index planes oscillate SLOWLY (coarse, long-range).")
print("Bottom: Each row is one rotation plane. Top rows change quickly,")
print("        bottom rows change slowly — like digits on an odometer.")

## 10. Long-Term Decay — Eq. 35

### WHAT
We analyze and visualize the **long-term decay** property of RoPE: the attention score between two tokens naturally decreases as their distance increases. This provides a built-in **recency bias**.

### WHY
In language, nearby words are usually more relevant to each other than distant ones. RoPE provides this inductive bias **for free**, without any learned parameters. The decay happens because the $d/2$ frequency components undergo **destructive interference** at large distances.

### HOW
The paper shows (Eq. 35) that for a query $\mathbf{q}$ at position $m$ and a key $\mathbf{k}$ at position $n$:

$$\langle R^d_{\Theta,m} \mathbf{q}, R^d_{\Theta,n} \mathbf{k} \rangle = \mathbf{q}^\top R^d_{\Theta, n-m} \mathbf{k} = \sum_{i=1}^{d/2} \left[ q_{2i-1} k_{2i-1} + q_{2i} k_{2i} \right] \cos((n-m)\theta_i)$$
$$+ \sum_{i=1}^{d/2} \left[ q_{2i-1} k_{2i} - q_{2i} k_{2i-1} \right] \sin((n-m)\theta_i)$$

As $|n-m|$ grows, the cosines and sines with different frequencies $\theta_i$ oscillate and **cancel out** (destructive interference), causing the overall sum to decay.

### WHERE
Section 3.4.3 and Figure 2 of the paper: *"It demonstrates that the inner product will decay as the relative distance increases."*

In [ ]:
# === Long-Term Decay Analysis ===

d = 64  # Head dimension
thetas = compute_theta_schedule(d)  # (d/2,) frequencies

# We'll analyze the decay in two ways:
# 1) The "pure" decay: what happens when q_i = k_i (identical content)
# 2) Random q, k: averaged over many samples

max_distance = 512
distances = torch.arange(max_distance, dtype=torch.float32)

# === Method 1: Pure decay (q = k, all entries = 1) ===
# When q_{2i-1} = k_{2i-1} = q_{2i} = k_{2i} = 1:
# dot product = sum_i 2*cos((n-m)*θ_i) + sum_i 0 = 2 * sum_i cos(Δ*θ_i)
# This is the sum of d/2 cosines at different frequencies.

# Sum of cosines for each distance
angles_matrix = torch.outer(distances, thetas)  # (max_dist, d/2)
sum_cos = torch.cos(angles_matrix).sum(dim=1)     # (max_dist,)
# Normalize: at distance 0, all cos terms = 1, sum = d/2
sum_cos_normalized = sum_cos / (d // 2)

# === Method 2: Random q, k ===
n_samples = 200
random_dots = torch.zeros(max_distance)

for _ in range(n_samples):
    q = torch.randn(d)  # Random query
    k = torch.randn(d)  # Random key
    
    for dist_idx, dist in enumerate(range(max_distance)):
        # Build rotation matrix for relative distance
        # Instead of full matrix, use the efficient formula
        dot = 0.0
        for i in range(d // 2):
            angle = dist * thetas[i].item()
            # From Eq. 35:
            # cos term: (q_{2i} * k_{2i} + q_{2i+1} * k_{2i+1}) * cos(Δθ_i)
            cos_part = (q[2*i] * k[2*i] + q[2*i+1] * k[2*i+1]) * math.cos(angle)
            # sin term: (q_{2i} * k_{2i+1} - q_{2i+1} * k_{2i}) * sin(Δθ_i)
            sin_part = (q[2*i] * k[2*i+1] - q[2*i+1] * k[2*i]) * math.sin(angle)
            dot += cos_part + sin_part
        random_dots[dist_idx] += dot

random_dots /= n_samples
# Normalize by the value at distance 0
random_dots_normalized = random_dots / random_dots[0]

# === Plot ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Pure decay (sum of cosines)
ax = axes[0]
ax.plot(distances.numpy(), sum_cos_normalized.numpy(), 'b-', linewidth=1.5)
ax.set_xlabel('Relative distance |m - n|', fontsize=12)
ax.set_ylabel('Normalized dot product', fontsize=12)
ax.set_title(f'Long-Term Decay: Sum of Cosines\n(d={d}, {d//2} frequencies)', fontsize=13)
ax.axhline(y=0, color='k', linewidth=0.5, linestyle='--')
ax.set_xlim(0, max_distance)

# Right: Envelope decay
ax = axes[1]
# Compute a moving envelope (max of absolute values in windows)
window_size = 10
envelope = []
for i in range(0, max_distance - window_size, window_size):
    chunk = sum_cos_normalized[i:i+window_size].abs().max().item()
    envelope.append((i + window_size // 2, chunk))

env_x, env_y = zip(*envelope)
ax.plot(distances.numpy()[:max_distance], sum_cos_normalized.numpy()[:max_distance].clip(-1, 1),
        'b-', alpha=0.3, linewidth=0.5)
ax.plot(env_x, env_y, 'r-', linewidth=2, label='Envelope (peak amplitude)')
ax.set_xlabel('Relative distance |m - n|', fontsize=12)
ax.set_ylabel('Amplitude', fontsize=12)
ax.set_title('Decay Envelope\n(reproducing Figure 2 from the paper)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, max_distance)

plt.tight_layout()
plt.show()

print("The attention contribution decays with distance due to destructive interference")
print("among the d/2 frequency components. This gives RoPE a natural recency bias.")
print(f"\nAt distance 0: normalized dot = {sum_cos_normalized[0]:.4f}")
print(f"At distance 50: normalized dot = {sum_cos_normalized[50]:.4f}")
print(f"At distance 200: normalized dot = {sum_cos_normalized[200]:.4f}")

## 11. RoPE in a Full Attention Layer

### WHAT
We implement a **complete Multi-Head Attention module with RoPE** and demonstrate it on a small sequence.

### WHY
To see how RoPE fits into the full Transformer pipeline. In standard multi-head attention, position encoding is added to the input. With RoPE, we instead **rotate** the queries and keys **after** projection but **before** the dot product. Crucially, the **values** are NOT rotated — only Q and K receive position information.

### HOW
The forward pass:
1. Project inputs: $Q = xW_Q$, $K = xW_K$, $V = xW_V$
2. Split into heads
3. **Apply RoPE** to Q and K (but NOT V)
4. Compute attention: $\text{softmax}(QK^\top / \sqrt{d_k}) V$
5. Concatenate heads and project output

### WHERE
Section 3.1, especially the overview: *"We propose to encode the position information by rotating the affine-transformed word embedding vector."* The full integration is shown in Section 3.4.

In [ ]:
# === Complete Multi-Head Attention with RoPE ===

class RotaryMultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention with Rotary Position Embedding (RoPE).
    
    This is the module that sits at the heart of every RoPE-based LLM.
    Key difference from standard MHA: RoPE is applied to Q and K AFTER
    projection, and values V receive NO position encoding.
    """
    
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int = 2048, base: float = 10000.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # Per-head dimension
        self.scale = math.sqrt(self.d_k)
        
        # Linear projections for Q, K, V, and output
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        # Precompute RoPE frequencies
        freqs_cis = precompute_freqs_cis(self.d_k, max_seq_len, base)
        # Register as buffer (not a parameter, but moves with the model to GPU)
        self.register_buffer('freqs_cis', freqs_cis)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """Forward pass.
        
        x:    (batch, seq_len, d_model)
        mask: (seq_len, seq_len) optional attention mask
        
        Returns: (output, attention_weights)
        """
        batch, seq_len, _ = x.shape
        
        # Step 1: Linear projections
        q = self.W_q(x)  # (batch, seq_len, d_model)
        k = self.W_k(x)
        v = self.W_v(x)
        
        # Step 2: Reshape to multi-head format
        # (batch, seq_len, d_model) → (batch, seq_len, n_heads, d_k)
        q = q.view(batch, seq_len, self.n_heads, self.d_k)
        k = k.view(batch, seq_len, self.n_heads, self.d_k)
        v = v.view(batch, seq_len, self.n_heads, self.d_k)
        
        # Step 3: Apply RoPE to Q and K (NOT V!)
        # This is the key step that distinguishes RoPE from other position encodings
        freqs = self.freqs_cis[:seq_len]  # (seq_len, d_k/2) complex
        q = apply_rotary_emb_complex(q, freqs)  # Rotate queries
        k = apply_rotary_emb_complex(k, freqs)  # Rotate keys
        # v is NOT rotated — values carry content only, not position
        
        # Step 4: Transpose for attention: (batch, n_heads, seq_len, d_k)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        # Step 5: Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale  # (batch, n_heads, seq_len, seq_len)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)  # (batch, n_heads, seq_len, seq_len)
        attn_output = torch.matmul(attn_weights, v)  # (batch, n_heads, seq_len, d_k)
        
        # Step 6: Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous()  # (batch, seq_len, n_heads, d_k)
        attn_output = attn_output.view(batch, seq_len, self.d_model)  # (batch, seq_len, d_model)
        
        # Step 7: Output projection
        output = self.W_o(attn_output)  # (batch, seq_len, d_model)
        
        return output, attn_weights


# === Demo: Run RoPE attention on a small sequence ===
d_model = 32
n_heads = 4
seq_len = 8
batch = 2

rope_attn = RotaryMultiHeadAttention(d_model, n_heads, max_seq_len=64)

# Random input (in practice, this would be token embeddings)
x = torch.randn(batch, seq_len, d_model)

# Causal mask (decoder-style: each token can only attend to itself and earlier tokens)
causal_mask = torch.tril(torch.ones(seq_len, seq_len))  # Lower triangular

output, attn_weights = rope_attn(x, mask=causal_mask)

print(f"=== RoPE Multi-Head Attention Demo ===")
print(f"Input shape:            {x.shape}  (batch, seq_len, d_model)")
print(f"Output shape:           {output.shape}  (batch, seq_len, d_model)")
print(f"Attention weights shape: {attn_weights.shape}  (batch, n_heads, seq_len, seq_len)")
print(f"")
print(f"Number of parameters: {sum(p.numel() for p in rope_attn.parameters()):,}")
print(f"Number of position parameters: 0 (RoPE has NO learnable position params!)")

In [ ]:
# === Visualize Attention Patterns with RoPE ===

fig, axes = plt.subplots(1, n_heads, figsize=(16, 4))

for head_idx in range(n_heads):
    ax = axes[head_idx]
    weights = attn_weights[0, head_idx].detach().numpy()  # (seq_len, seq_len)
    im = ax.imshow(weights, cmap='Blues', vmin=0, aspect='equal')
    ax.set_title(f'Head {head_idx}', fontsize=12)
    ax.set_xlabel('Key position')
    if head_idx == 0:
        ax.set_ylabel('Query position')

plt.suptitle('RoPE Attention Weights (Causal Mask, Batch 0)', fontsize=14, y=1.05)
plt.colorbar(im, ax=axes, label='Attention weight', shrink=0.8)
plt.tight_layout()
plt.show()

print("The lower-triangular pattern comes from the causal mask.")
print("Different heads learn different attention patterns,")
print("but RoPE ensures they all have position awareness.")

In [ ]:
# === Demonstrate: RoPE attention scores depend on RELATIVE position ===

# Create a scenario where we place the SAME two tokens at different absolute positions
# but the same relative distance apart, and verify the attention score is identical.

d_model = 16
n_heads = 2
d_k = d_model // n_heads  # = 8

# Two specific token embeddings
token_a = torch.randn(d_model)
token_b = torch.randn(d_model)

# Simple Q and K projections (identity for clarity)
W_q = torch.eye(d_model)
W_k = torch.eye(d_model)

# Project
q_raw = W_q @ token_a  # (d_model,)
k_raw = W_k @ token_b  # (d_model,)

# For one head, take the first d_k dimensions
q_head = q_raw[:d_k]
k_head = k_raw[:d_k]

# Apply RoPE at different absolute positions but SAME relative distance
freqs = precompute_freqs_cis(d_k, max_seq_len=200)

print("=== Relative Position Invariance ===")
print(f"Testing: same content, same relative distance, different absolute positions\n")

relative_dist = 5
test_positions = [(3, 3 + relative_dist), (20, 20 + relative_dist),
                  (100, 100 + relative_dist), (150, 150 + relative_dist)]

scores = []
for m, n in test_positions:
    # Reshape for apply_rotary_emb_complex: (1, 1, 1, d_k)
    q_in = q_head.unsqueeze(0).unsqueeze(0).unsqueeze(0)  # (1, 1, 1, d_k)
    k_in = k_head.unsqueeze(0).unsqueeze(0).unsqueeze(0)
    
    # Apply RoPE
    q_rot = apply_rotary_emb_complex(q_in, freqs[m:m+1])  # Use position m
    k_rot = apply_rotary_emb_complex(k_in, freqs[n:n+1])  # Use position n
    
    # Compute dot product
    score = torch.sum(q_rot * k_rot).item()
    scores.append(score)
    print(f"  m={m:3d}, n={n:3d} (relative dist={relative_dist}) → dot = {score:.8f}")

max_diff = max(scores) - min(scores)
print(f"\nMax variation across positions: {max_diff:.2e}")
print(f"All scores equal? {max_diff < 1e-5}  ✓")
print(f"\nThis confirms: RoPE attention scores depend only on RELATIVE position!")

## 12. Length Extrapolation Demo

### WHAT
We demonstrate one of RoPE's most important practical advantages: the ability to **extrapolate** to sequence lengths longer than those seen during training. We compare RoPE against learned position embeddings.

### WHY
**Learned position embeddings** are a lookup table of size $L_{\max} \times d$. If the model was trained with $L_{\max} = 512$, it literally cannot process position 513 — there's no embedding for it. 

**RoPE**, on the other hand, is a **continuous function** of position: $R(m\theta)$ is defined for any $m \in \mathbb{R}$. While the model's performance may degrade for very long contexts (the rotation angles become unfamiliar), it **doesn't crash** and **degrades gracefully**.

### HOW
We create a simple task (predicting attention patterns), train with short sequences, then test with longer ones. We measure how the attention score distribution changes.

### WHERE
Section 4.2 and Table 2 of the paper discuss length extrapolation experiments.

In [ ]:
# === Length Extrapolation: RoPE vs Learned Position Embeddings ===

class LearnedPosAttention(nn.Module):
    """Simple attention with LEARNED position embeddings (like GPT-2/BERT)."""
    
    def __init__(self, d_model: int, max_seq_len: int):
        super().__init__()
        # Learned position embedding table — FIXED size!
        self.pos_emb = nn.Embedding(max_seq_len, d_model)  # Can't go beyond max_seq_len
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
    
    def get_scores(self, x: torch.Tensor) -> torch.Tensor:
        """Compute attention scores with learned positional encoding.
        x: (batch, seq_len, d_model)
        """
        batch, seq_len, d = x.shape
        if seq_len > self.pos_emb.num_embeddings:
            raise RuntimeError(
                f"Sequence length {seq_len} exceeds maximum {self.pos_emb.num_embeddings}!"
                f" Learned embeddings CANNOT extrapolate!"
            )
        
        # Add position embedding (additive)
        positions = torch.arange(seq_len, device=x.device)
        x_pos = x + self.pos_emb(positions).unsqueeze(0)  # (batch, seq_len, d)
        
        q = self.W_q(x_pos)
        k = self.W_k(x_pos)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)
        return scores


class RoPEAttentionSimple(nn.Module):
    """Simple attention with RoPE — NO maximum sequence length!"""
    
    def __init__(self, d_model: int):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.d_model = d_model
    
    def get_scores(self, x: torch.Tensor) -> torch.Tensor:
        """Compute attention scores with RoPE.
        x: (batch, seq_len, d_model)
        """
        batch, seq_len, d = x.shape
        # RoPE frequencies can be computed for ANY length!
        freqs = precompute_freqs_cis(d, seq_len)  # (seq_len, d/2)
        
        q = self.W_q(x)  # (batch, seq_len, d)
        k = self.W_k(x)
        
        # Add a dummy "heads" dimension: (batch, seq_len, 1, d)
        q = q.unsqueeze(2)
        k = k.unsqueeze(2)
        
        # Apply RoPE
        q = apply_rotary_emb_complex(q, freqs)
        k = apply_rotary_emb_complex(k, freqs)
        
        # Remove dummy dimension and compute scores
        q = q.squeeze(2)
        k = k.squeeze(2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)
        return scores


# === Test extrapolation ===
d_model = 16
train_length = 32  # "Trained" with this length

learned_attn = LearnedPosAttention(d_model, max_seq_len=train_length)
rope_attn_simple = RoPEAttentionSimple(d_model)

test_lengths = [16, 32, 48, 64, 96, 128]

print("=== Length Extrapolation Test ===")
print(f"Training max length: {train_length}\n")
print(f"{'Length':>8} | {'Learned PE':>20} | {'RoPE':>20}")
print("-" * 55)

for length in test_lengths:
    x = torch.randn(1, length, d_model)
    
    # Try learned PE
    try:
        scores_learned = learned_attn.get_scores(x)
        learned_status = f"OK (mean={scores_learned.mean():.3f})"
    except RuntimeError as e:
        learned_status = "CRASH! ✗"
    
    # Try RoPE
    scores_rope = rope_attn_simple.get_scores(x)
    rope_status = f"OK (mean={scores_rope.mean():.3f})"
    
    print(f"{length:>8} | {learned_status:>20} | {rope_status:>20}")

print()
print("CONCLUSION: Learned PE crashes beyond max_seq_len.")
print("RoPE handles ANY length because it's a continuous function of position.")

In [ ]:
# === Visualize: How RoPE attention scores change with extrapolation ===

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

d_model = 32
rope_model = RoPEAttentionSimple(d_model)

# Fixed content (same tokens replicated to different lengths)
base_tokens = torch.randn(1, 16, d_model)  # 16 base tokens

test_configs = [
    (16, 'Within training range'),
    (48, '1.5x extrapolation'),
    (96, '3x extrapolation'),
]

for ax, (length, title) in zip(axes, test_configs):
    # Repeat tokens to fill the length
    repeats = (length + 15) // 16
    x = base_tokens.repeat(1, repeats, 1)[:, :length, :]
    
    scores = rope_model.get_scores(x)
    attn = F.softmax(scores[0], dim=-1).detach().numpy()
    
    im = ax.imshow(attn, cmap='Blues', aspect='auto')
    ax.set_title(f'{title}\n(length={length})', fontsize=11)
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')

plt.suptitle('RoPE Attention Patterns Under Length Extrapolation', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("RoPE maintains structured attention patterns even at 3x the training length.")
print("The patterns degrade gradually rather than catastrophically.")

## 13. Comparison: Learned vs Sinusoidal vs RoPE

### WHAT
We provide a comprehensive side-by-side comparison of the three main position encoding approaches: **learned absolute**, **sinusoidal**, and **RoPE**.

### WHY
To understand why RoPE has become the standard, it helps to see how each approach works and where they differ in terms of:
- Extra parameters
- Where position information enters the computation
- Ability to encode relative positions
- Length extrapolation capability

### HOW
We implement all three and show their effect on the same input.

### WHERE
Section 2 (Related Work) and Section 4 (Experiments) of the paper compare against various baselines.

In [ ]:
# === Side-by-Side Comparison of Position Encoding Methods ===

d_model = 32
seq_len = 64

# --- 1. Sinusoidal Position Encoding ---
def build_sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
    """Build the sinusoidal PE matrix from Vaswani et al. (2017).
    PE(pos, 2i)   = sin(pos / 10000^(2i/d))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
    
    Returns: (seq_len, d_model)
    """
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)  # (seq_len, 1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32) * -(math.log(10000.0) / d_model)
    )  # (d_model/2,)
    
    pe[:, 0::2] = torch.sin(position * div_term)  # Even dimensions
    pe[:, 1::2] = torch.cos(position * div_term)  # Odd dimensions
    return pe


# --- 2. Learned Position Encoding ---
learned_pe = nn.Embedding(seq_len, d_model)  # Lookup table

# --- 3. RoPE (already implemented) ---
rope_freqs = precompute_freqs_cis(d_model, seq_len)

# === Visualize the position encodings ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sinusoidal
sin_pe = build_sinusoidal_pe(seq_len, d_model)
im1 = axes[0].imshow(sin_pe.numpy(), cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[0].set_title('Sinusoidal PE\n(Vaswani et al., 2017)', fontsize=12)
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Position')

# Learned (random init — in practice these are trained)
with torch.no_grad():
    positions = torch.arange(seq_len)
    lrn_pe = learned_pe(positions)
im2 = axes[1].imshow(lrn_pe.numpy(), cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[1].set_title('Learned PE\n(GPT-2, BERT)', fontsize=12)
axes[1].set_xlabel('Dimension')

# RoPE (show the cos component)
thetas = compute_theta_schedule(d_model)
pos_range = torch.arange(seq_len, dtype=torch.float32)
rope_cos = torch.cos(torch.outer(pos_range, thetas))  # (seq_len, d/2)
# Interleave cos and sin for visualization
rope_sin = torch.sin(torch.outer(pos_range, thetas))
rope_vis = torch.zeros(seq_len, d_model)
rope_vis[:, 0::2] = rope_cos
rope_vis[:, 1::2] = rope_sin
im3 = axes[2].imshow(rope_vis.numpy(), cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[2].set_title('RoPE Rotation Factors\n(Su et al., 2021)', fontsize=12)
axes[2].set_xlabel('Dimension')

plt.colorbar(im3, ax=axes, label='Value', shrink=0.8)
plt.tight_layout()
plt.show()

# === Summary Table ===
print("\n" + "=" * 80)
print(f"{'Property':<30} {'Sinusoidal':>15} {'Learned':>15} {'RoPE':>15}")
print("=" * 80)
print(f"{'Extra parameters':<30} {'0':>15} {f'{seq_len * d_model:,}':>15} {'0':>15}")
print(f"{'Where position enters':<30} {'Addition':>15} {'Addition':>15} {'Rotation':>15}")
print(f"{'Encodes relative position':<30} {'No':>15} {'No':>15} {'Yes':>15}")
print(f"{'Length extrapolation':<30} {'Partial':>15} {'No':>15} {'Yes':>15}")
print(f"{'Theoretically derived':<30} {'No':>15} {'No':>15} {'Yes':>15}")
print(f"{'Applied to':<30} {'Input':>15} {'Input':>15} {'Q, K only':>15}")
print(f"{'Values get position info':<30} {'Yes':>15} {'Yes':>15} {'No':>15}")
print("=" * 80)

In [ ]:
# === Visualize: Dot product similarity as a function of position distance ===
# This shows how each encoding method handles relative position.

d_model = 64
max_pos = 256

# --- Sinusoidal PE: dot product between PE(m) and PE(n) ---
sin_pe = build_sinusoidal_pe(max_pos, d_model)  # (max_pos, d_model)

# Fix position 0, compute dot products with all other positions
ref_pos = 0
sin_dots = torch.matmul(sin_pe, sin_pe[ref_pos])  # (max_pos,)
sin_dots_normalized = sin_dots / sin_dots[ref_pos]  # Normalize

# --- Learned PE: dot product (random, so no meaningful pattern) ---
learned_pe_large = nn.Embedding(max_pos, d_model)
with torch.no_grad():
    positions = torch.arange(max_pos)
    lrn_vecs = learned_pe_large(positions)
    lrn_dots = torch.matmul(lrn_vecs, lrn_vecs[ref_pos])
    lrn_dots_normalized = lrn_dots / lrn_dots[ref_pos]

# --- RoPE: use the sum-of-cosines formula ---
thetas = compute_theta_schedule(d_model)
pos_range = torch.arange(max_pos, dtype=torch.float32)
rope_decay = torch.cos(torch.outer(pos_range, thetas)).sum(dim=1) / (d_model // 2)

# === Plot ===
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

ax.plot(pos_range.numpy(), sin_dots_normalized.numpy(), 'b-', linewidth=2,
        label='Sinusoidal PE (dot with pos 0)', alpha=0.8)
ax.plot(pos_range.numpy(), lrn_dots_normalized.detach().numpy(), 'g-', linewidth=1,
        label='Learned PE (random init, dot with pos 0)', alpha=0.5)
ax.plot(pos_range.numpy(), rope_decay.numpy(), 'r-', linewidth=2,
        label='RoPE (relative decay)', alpha=0.8)

ax.axhline(y=0, color='k', linewidth=0.5, linestyle='--')
ax.set_xlabel('Distance from reference position', fontsize=12)
ax.set_ylabel('Normalized similarity', fontsize=12)
ax.set_title('Position Encoding Similarity vs Distance', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, max_pos)

plt.tight_layout()
plt.show()

print("Sinusoidal: Regular oscillation, does NOT decay — no recency bias.")
print("Learned:    Random pattern (untrained) — no structure until learned.")
print("RoPE:       Natural decay with distance — built-in recency bias.")

## 14. Key Takeaways

---

### Why RoPE Won: A Summary

1. **Theoretically Derived, Not Guessed**: RoPE is the unique solution to the relativity constraint (Eq. 11). It's not a heuristic — it's derived from first principles.

2. **Multiplicative, Not Additive**: Position is encoded by **rotating** Q and K, not by adding a bias. This means:
   - Values are content-only (no position leakage)
   - The position signal doesn't interfere with the content signal

3. **Zero Extra Parameters**: Unlike learned embeddings, RoPE adds **no** trainable parameters. The rotation frequencies are computed from a fixed formula.

4. **Natural Relative Position Encoding**: The dot product $\langle R_m q, R_n k \rangle$ depends only on $m - n$. This is the most natural way to capture that "3 tokens apart" should mean the same thing regardless of absolute position.

5. **Length Extrapolation**: Because RoPE is a continuous function of position, models can process sequences longer than those seen during training (with graceful degradation).

6. **Long-Term Decay**: The interference pattern of $d/2$ frequency components creates a natural decay with distance, providing a recency bias without any explicit design.

7. **Computationally Efficient**: The element-wise formulation (Eq. 34) makes RoPE essentially free — just element-wise multiply and add.

---

### Who Uses RoPE

| Model | Organization | Year |
|-------|-------------|------|
| GPT-NeoX | EleutherAI | 2022 |
| PaLM | Google | 2022 |
| LLaMA / LLaMA 2 / LLaMA 3 | Meta | 2023-2024 |
| Mistral / Mixtral | Mistral AI | 2023 |
| Qwen / Qwen2 | Alibaba | 2023-2024 |
| DeepSeek / DeepSeek-V2 | DeepSeek | 2024 |
| Gemma | Google | 2024 |
| Phi-3 | Microsoft | 2024 |

---

### The Key Equations to Remember

1. **Relativity Constraint** (Eq. 11): $\langle f_q(x_m, m), f_k(x_n, n) \rangle = g(x_m, x_n, m-n)$

2. **2D Solution** (Eq. 12): $f(x, m) = (Wx) e^{im\theta}$

3. **Frequency Schedule** (Eq. 16): $\theta_i = 10000^{-2(i-1)/d}$

4. **Efficient Implementation** (Eq. 34): $R_m x = x \odot \cos(m\theta) + \text{rotate\_half}(x) \odot \sin(m\theta)$

---

### References

- **Original Paper**: Su, J., Lu, Y., Pan, S., Murtadha, A., Wen, B., & Liu, Y. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding.* arXiv:2104.09864.
- **Vaswani et al. (2017)**: *Attention Is All You Need.* NeurIPS 2017. (Original sinusoidal PE)
- **Shaw et al. (2018)**: *Self-Attention with Relative Position Representations.* NAACL 2018.
- **Press et al. (2022)**: *ALiBi: Train Short, Test Long.* ICLR 2022. (Alternative: linear bias)